# Análisis Exploratorio de Datos (EDA) - Airbnb Bogotá

Exploración de las colecciones **Listings**, **Reviews** y **Calendar** almacenadas en MongoDB local, con el objetivo de identificar problemas de calidad y definir las transformaciones del pipeline ETL (`src/transformacion.py`).

In [1]:
# Celda 1: Importación de librerías, conexión a MongoDB y carga de colecciones
import os

import pandas as pd
import pymongo
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017/')
MONGO_DB = os.getenv('MONGO_DB', 'airbnb')
COLECCIONES = ['Listings', 'Reviews', 'Calendar']
# Calendar suele tener millones de filas (listings x 365 días); limitar para el EDA.
LIMITE_CALENDAR = 500_000

try:
    client = pymongo.MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    db = client[MONGO_DB]
    print(f'Conectado a MongoDB. Colecciones disponibles: {db.list_collection_names()}')
except pymongo.errors.PyMongoError as e:
    raise RuntimeError(f'No fue posible conectar a MongoDB: {e}')

dataframes = {}
for nombre in COLECCIONES:
    try:
        cursor = db[nombre].find({}, {'_id': 0})
        if nombre == 'Calendar':
            cursor = cursor.limit(LIMITE_CALENDAR)
        dataframes[nombre] = pd.DataFrame(list(cursor))
        print(f'{nombre}: {len(dataframes[nombre]):,} registros cargados')
    except pymongo.errors.PyMongoError as e:
        print(f'Error al cargar {nombre}: {e}')

df_listings = dataframes.get('Listings', pd.DataFrame())
df_reviews = dataframes.get('Reviews', pd.DataFrame())
df_calendar = dataframes.get('Calendar', pd.DataFrame())

Conectado a MongoDB. Colecciones disponibles: ['reviews', 'calendar', 'listings']
Listings: 0 registros cargados
Reviews: 0 registros cargados
Calendar: 0 registros cargados


In [ ]:
# Celda 2: Entendimiento general de cada DataFrame
for nombre, df in dataframes.items():
    print('=' * 80)
    print(f'{nombre} -> {df.shape[0]:,} registros | {df.shape[1]} columnas')
    print('=' * 80)
    display(df.head())
    df.info()
    print()

In [ ]:
# Celda 3: Calidad de datos - nulos y duplicados
def contar_duplicados(df):
    """Cuenta duplicados convirtiendo a texto las columnas con listas/diccionarios (no hashables)."""
    comparacion = df.copy()
    for col in comparacion.select_dtypes(include='object').columns:
        if comparacion[col].map(lambda v: isinstance(v, (list, dict))).any():
            comparacion[col] = comparacion[col].astype(str)
    return int(comparacion.duplicated().sum())

for nombre, df in dataframes.items():
    print('=' * 80)
    print(f'{nombre}')
    print('=' * 80)
    nulos = df.isnull().sum()
    resumen = pd.DataFrame({
        'nulos': nulos,
        'porcentaje_%': (nulos / max(len(df), 1) * 100).round(2)
    }).sort_values('nulos', ascending=False)
    display(resumen[resumen['nulos'] > 0])
    print(f'Registros duplicados: {contar_duplicados(df):,}\n')

In [ ]:
# Celda 4: Análisis de atípicos (outliers) en Listings y exportación a docs/img/
from pathlib import Path
CARPETA_IMG = Path('../docs/img')
CARPETA_IMG.mkdir(parents=True, exist_ok=True)

df_box = df_listings.copy()

# price viene como texto (ej. '$120,000.00'): se limpia solo para graficar.
if 'price' in df_box.columns and not pd.api.types.is_numeric_dtype(df_box['price']):
    df_box['price'] = pd.to_numeric(
        df_box['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False),
        errors='coerce'
    )

variables = [v for v in ['price', 'minimum_nights', 'availability_365'] if v in df_box.columns]
for v in variables:
    df_box[v] = pd.to_numeric(df_box[v], errors='coerce')

fig, axes = plt.subplots(1, len(variables), figsize=(6 * len(variables), 5))
axes = [axes] if len(variables) == 1 else axes
for ax, v in zip(axes, variables):
    sns.boxplot(y=df_box[v], ax=ax, color='skyblue')
    ax.set_title(f'Boxplot de {v}')
plt.tight_layout()
ruta_boxplots = CARPETA_IMG / 'boxplots_outliers.png'
plt.savefig(ruta_boxplots, dpi=300, bbox_inches='tight')
print(f'Gráfica guardada en: {ruta_boxplots.resolve()}')
plt.show()

# Gráfica de Distribución de Precios
if 'price' in df_box.columns:
    p99 = df_box['price'].quantile(0.99)
    plt.figure(figsize=(9, 5))
    sns.histplot(df_box[df_box['price'] <= p99]['price'], kde=True, color='teal', bins=35)
    plt.title('Distribución de Precios en COP (Filtrado hasta Percentil 99)')
    plt.xlabel('Precio (COP)')
    plt.ylabel('Frecuencia')
    plt.tight_layout()
    ruta_dist_precios = CARPETA_IMG / 'distribucion_precios.png'
    plt.savefig(ruta_dist_precios, dpi=300, bbox_inches='tight')
    print(f'Gráfica guardada en: {ruta_dist_precios.resolve()}')
    plt.show()

# Cuantificación de outliers con el criterio IQR (1.5 x IQR)
for v in variables:
    q1, q3 = df_box[v].quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df_box[v] < lim_inf) | (df_box[v] > lim_sup)).sum()
    print(f'{v}: {n_out:,} outliers | límites IQR [{lim_inf:,.2f}, {lim_sup:,.2f}] | máx = {df_box[v].max():,.2f}')


## Celda 5: Hallazgos, inconsistencias y transformaciones necesarias

> Validar las cifras con la salida de las celdas anteriores; los puntos siguientes resumen los patrones típicos de los datasets de Inside Airbnb para Bogotá.

### Hallazgos principales
- **Listings** es la colección central: cada anuncio tiene su ubicación, tipo de habitación, precio, noches mínimas, disponibilidad y métricas de reseñas.
- **Reviews** y **Calendar** se relacionan con Listings mediante `listing_id`. Calendar es la colección más grande (≈ anuncios × 365 días).
- Los precios están en **pesos colombianos (COP)**, con una distribución muy sesgada a la derecha.

### Inconsistencias encontradas
1. **`price` almacenado como texto** con símbolo `$` y separador de miles `,` (ej. `"$150,000.00"`), lo que impide cálculos numéricos.
2. **Valores nulos**: columnas casi vacías (ej. `license`, `calendar_updated`, `neighbourhood_group_cleansed`), nulos en `price`, `review_scores_*`, `host_*`, `bedrooms`/`beds`, y comentarios vacíos en Reviews.
3. **Outliers**: precios extremos (muy por encima del límite IQR o iguales a 0), `minimum_nights` con valores irreales (ej. > 365) y `availability_365` concentrada en 0 o 365 (anuncios inactivos o sin gestión).
4. **Fechas como texto** (`date`, `last_scraped`, `first_review`, `last_review`), sin tipo datetime.
5. **Posibles duplicados** derivados de importaciones repetidas a MongoDB.
6. **Caracteres especiales / de control** en los comentarios de Reviews que pueden romper la exportación a Excel.

### Transformaciones necesarias (`src/transformacion.py`)
| Problema | Método | Acción |
|---|---|---|
| Duplicados y nulos | `limpiar_nulos_y_duplicados` | Eliminar duplicados, eliminar columnas con > 90 % de nulos, descartar filas sin `price`/`date` e imputar mediana (numéricos) o `'Sin información'` (texto). |
| Precio como texto | `normalizar_precio` | Quitar `$` y `,` y convertir a `float`. |
| Fechas como texto | `procesar_fechas` | Convertir a datetime `YYYY-MM-DD` y derivar año, mes, día y trimestre. |
| Distribución sesgada de precios | `categorizar_precios` | Segmentar con `pd.cut` en **Económico / Estándar / Premium** usando percentiles 33 y 66 (robusto frente a outliers). |
| Caracteres ilegales y campos anidados | `Carga` | Sanitizar texto y serializar listas/diccionarios antes de cargar en SQLite/Excel. |